In [18]:
import os
import scanpy as sc
import celltypist
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import tarfile
import urllib.request
import shutil
import glob
import ssl

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Scanpy settings
sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

print("Environment setup complete.")

Environment setup complete.


In [19]:
import os
import tarfile
import urllib.request
import shutil
import glob
import ssl

# --- 1. Setup Data Directory ---
os.makedirs("data", exist_ok=True)

# --- 2. Download Whitelist ---
url = "https://github.com/f0t1h/3M-february-2018/raw/refs/heads/master/3M-february-2018.txt.gz"
whitelist_path = "data/whitelist.txt.gz"

# SSL Context for Mac/Linux compatibility
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

if not os.path.exists(whitelist_path):
    print(f"Downloading whitelist...")
    try:
        with urllib.request.urlopen(url, context=ssl_context) as response, open(whitelist_path, 'wb') as out_file:
            shutil.copyfileobj(response, out_file)
        os.system(f"gunzip -f {whitelist_path}") # Unzip immediately
        print("Whitelist ready.")
    except Exception as e:
        print(f"Whitelist download failed: {e}")

# --- 3. Extract Input Data (Verbose) ---
tar_filename = "toy_read_ref_set.tar.gz"
if os.path.exists(tar_filename):
    print(f"--- Extracting {tar_filename} ---")
    try:
        with tarfile.open(tar_filename, "r:*") as tar:
            # Print contents so we can debug if names are weird
            names = tar.getnames()
            print(f"Files in archive: {names}")
            tar.extractall()
        print("Extraction complete.")
    except Exception as e:
        print(f"Extraction failed: {e}")
else:
    print(f"WARNING: {tar_filename} not found. Assuming files are already present.")

# --- 4. Robust File Finding & Renaming ---
print("--- searching for files ---")

def safe_rename(src, dst):
    if os.path.abspath(src) == os.path.abspath(dst):
        return
    print(f"Renaming {src} -> {dst}")
    shutil.move(src, dst)

# Recursive file walker to find inputs regardless of folder structure
all_files = []
for root, dirs, files in os.walk("."):
    for f in files:
        # Skip hidden files and the data folder we just made
        if not f.startswith(".") and "data/" not in root:
            all_files.append(os.path.join(root, f))

# Find Genome (Look for .fa or .fasta, case insensitive)
genome_candidates = [f for f in all_files if f.lower().endswith(('.fa', '.fasta'))]
if genome_candidates:
    # Pick largest file to avoid small metadata files
    best_genome = max(genome_candidates, key=os.path.getsize)
    safe_rename(best_genome, "genome.fa")

# Find GTF
gtf_candidates = [f for f in all_files if f.lower().endswith('.gtf')]
if gtf_candidates:
    best_gtf = max(gtf_candidates, key=os.path.getsize)
    safe_rename(best_gtf, "genes.gtf")

# Find FASTQs (Look for .fq.gz, .fastq.gz, .fq, .fastq)
# Exclude the whitelist and the tarball itself
fastq_candidates = [
    f for f in all_files 
    if f.lower().endswith(('.fastq.gz', '.fq.gz', '.fastq', '.fq'))
    and "whitelist" not in f
    and "toy_read_ref_set" not in f
]

# Sort to ensure Read 1 comes before Read 2
fastq_candidates = sorted(fastq_candidates)
print(f"FASTQ Candidates found: {fastq_candidates}")

if len(fastq_candidates) >= 2:
    safe_rename(fastq_candidates[0], "read1.fastq.gz")
    safe_rename(fastq_candidates[1], "read2.fastq.gz")
elif os.path.exists("read1.fastq.gz") and os.path.exists("read2.fastq.gz"):
    print("read1 and read2 already exist.")
else:
    print("ERROR: Could not locate 2 FASTQ files. Dumping file list for debug:")
    print(all_files)

# Final check
print("--- Final Directory State ---")
os.system("ls -lh genome.fa genes.gtf read*.fastq.gz 2>/dev/null")

Whitelist ready.
Extracting toy_read_ref_set.tar.gz...
Renamed: toy_ref_read/toy_human_ref/fasta/genome.fa -> genome.fa
Renamed: toy_ref_read/toy_human_ref/genes/genes.gtf -> genes.gtf
FASTQ Candidates found: []
ERROR: Could not locate 2 FASTQ files. Check the candidates list above.
--- Current Directory Content ---
-rw-rw-r--  1 lev  staff   435K Nov 10  2022 genes.gtf
-rw-rw-r--  1 lev  staff   175M Nov 10  2022 genome.fa


256

In [17]:
%%bash
set -e

# --- Configuration ---
GENOME="genome.fa"
GTF="genes.gtf"
R1="read1.fastq.gz"
R2="read2.fastq.gz"
IDX="data/salmon_index"
MAP_OUT="data/alevin_out"

echo "--- Verifying Inputs for Salmon ---"
if [[ ! -f "$R1" || ! -f "$R2" ]]; then
    echo "ERROR: FASTQ files missing. Cell 2 failed to rename them."
    exit 1
fi

# --- Step 1: Generate t2g ---
echo "Generating t2g.tsv..."
grep 'transcript_id' $GTF | \
awk -F';' '{print $1, $3}' | \
sed 's/transcript_id "//' | sed 's/"; gene_id "/\t/' | sed 's/"//' > data/t2g.tsv

# --- Step 2: Build Index ---
echo "Building Salmon index..."
salmon index -t $GENOME -i $IDX -p 2

# --- Step 3: Run Alevin ---
echo "Running Salmon Alevin..."
salmon alevin -l ISR -1 $R1 -2 $R2 \
  -i $IDX \
  --tgMap data/t2g.tsv \
  --output $MAP_OUT \
  --rad --sketch \
  -p 2

echo "Salmon pipeline complete."

Generating t2g map...
Building Salmon index...
zsh:1: command not found: salmon
Running Salmon Alevin mapping...
zsh:1: command not found: salmon


In [ ]:
%%bash
set -e  # Stop immediately on error

MAP_OUT="data/alevin_out"
QUANT_OUT="data/fry_quant"
WHITELIST="data/whitelist.txt"
T2G="data/t2g.tsv"

# --- Step 1: Generate Permit List ---
echo "Generating permit list..."
alevin-fry generate-permit-list -d forward -i $MAP_OUT -o $QUANT_OUT -u $WHITELIST

# --- Step 2: Collate ---
echo "Collating records..."
alevin-fry collate -i $QUANT_OUT -r $MAP_OUT -t 2

# --- Step 3: Quantify ---
echo "Quantifying..."
alevin-fry quant -i $QUANT_OUT -o $QUANT_OUT/res -t 2 -r cr-like -m $T2G --use-mtx

# --- Step 4: Verify and Compress Output ---
RES_DIR="$QUANT_OUT/res/alevin"
echo "Checking output in $RES_DIR..."
ls -F $RES_DIR

# Check if matrix.mtx exists (uncompressed) and zip it if needed
if [ -f "$RES_DIR/matrix.mtx" ]; then
    echo "Compressing matrix.mtx to matrix.mtx.gz..."
    gzip -f "$RES_DIR/matrix.mtx"
fi

# Verify the final file exists for Scanpy
if [ ! -f "$RES_DIR/matrix.mtx.gz" ]; then
    echo "ERROR: matrix.mtx.gz not found in $RES_DIR!"
    exit 1
fi

echo "SUCCESS: Matrix generated and verified."

In [ ]:
# 1. Load Data
print("Loading count matrix...")
# We point to the output folder from the previous step
adata = sc.read_10x_mtx(
    'data/fry_quant/res/alevin',
    var_names='gene_symbols', 
    cache=True
)

# 2. Quality Control (QC)
# Calculate mitochondrial content
adata.var['mt'] = adata.var_names.str.startswith('MT-') 
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

print(f"Cells before filtering: {adata.n_obs}")

# Filter cells/genes
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
# Filter high MT content
adata = adata[adata.obs.pct_counts_mt < 5, :]

print(f"Cells after filtering: {adata.n_obs}")

# 3. Normalization & Log Transformation
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# 4. Dimensionality Reduction & Clustering
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)
sc.tl.leiden(adata)

# Plot Clustering
sc.pl.umap(adata, color=['leiden'], title="Leiden Clustering", show=True)

In [ ]:
# 1. Load Model
print("Loading CellTypist model...")
# Downloads 'Immune_All_Low.pkl' if not present
model = celltypist.models.Model.load(model='Immune_All_Low.pkl')

# 2. Annotate
print("Annotating cells...")
predictions = celltypist.annotate(adata, model='Immune_All_Low.pkl', majority_voting=True)

# 3. Store results
adata.obs['cell_type'] = predictions.predicted_labels['predicted_labels']
adata.obs['conf_score'] = predictions.predicted_labels['conf_score']

# 4. Plot Annotation
sc.pl.umap(adata, color=['cell_type'], title="CellTypist Annotation", legend_loc='on data')

print("\nDetected Cell Types:")
print(adata.obs['cell_type'].value_counts())